In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import TfidfTransformer

# Initialize a matrix for the 200x200 grid (40,000 cells) and 85 POI categories
# Use mesh_x and mesh_y (1-200) to create a flattened index
poi_matrix = np.zeros((40000, 85))
directory = "cell_POIcat.csv/"

for filename in os.listdir(directory):
    try:
        parts = filename.split(',')
        if len(parts) == 4:
            x, y, cat_id, count = map(int, parts)
            # Flatten 2D grid (1-200) to 1D index (0-39999)
            grid_idx = (x - 1) * 200 + (y - 1)
            poi_matrix[grid_idx, cat_id - 1] = count
    except ValueError:
        continue

poi_df = pd.DataFrame(poi_matrix)

In [ ]:
tfidf = TfidfTransformer()
weighted_poi = tfidf.fit_transform(poi_df)

# LDA for Functional Zones
# Choose 5 zones: e.g., Residential, Business, Industrial, Transit, Green Space
lda = LatentDirichletAllocation(n_components=5, random_state=42)
zone_distributions = lda.fit_transform(weighted_poi)

# Save the results back to a lookup table
poi_df['functional_zone'] = zone_distributions.argmax(axis=1)
poi_df['grid_id'] = range(40000)

In [ ]:
poi_df["functional_zone"].value_counts()

dominant_zone
0    23778
1     7776
3     3297
4     2946
2     2203
Name: count, dtype: int64